# 02 · 线性混合模型、岭回归与 LOCO Linear Mixed Model, Ridge & LOCO

> **能力标签**：GB3（群体结构与混合模型 / Population Structure & Mixed Models）

## 目标 Objectives

完成本课后，你应该能够：

1. 阐述**线性混合模型（LMM, linear mixed model）**的直觉：固定效应（fixed effects）+ 多基因随机效应（polygenic random effects）。
2. 理解**全基因组岭回归（whole-genome ridge regression）**如何近似 LMM 的多基因成分（polygenic component）。
3. 解释**留一染色体（LOCO, Leave-One-Chromosome-Out）**方法如何避免**邻近污染（proximal contamination）**——即防止待检变异被自身所在染色体的岭回归"解释掉"。
4. 概述 **REGENIE 两步法（REGENIE two-step）**的思路，理解大规模 GWAS 中的实现策略。
5. 为 **b8-ridge-lmm** 和 **b8-loco** 作业做好准备：实现 `ridge_predict` 和 `loco_residualize`。

> 对应能力：**GB3**


## 概念讲解 Concepts

### 1 · 线性混合模型直觉 LMM Intuition

标准 GWAS 用线性回归 $y = X\beta + \varepsilon$，假设残差 $\varepsilon \sim \mathcal{N}(0, \sigma^2_e I)$（独立同分布）。
但在有群体分层或亲缘关系（relatedness）的样本中，个体之间存在遗传相关性（genetic correlation），
残差并非独立，会导致统计量膨胀。

**LMM** 在模型中显式引入**多基因随机效应（polygenic random effect）** $u$：

$$y = X\beta + g + \varepsilon$$

其中：
- $X\beta$：固定效应（fixed effects），包含截距、协变量、待检 SNP
- $g$：多基因随机效应（polygenic term），$g \sim \mathcal{N}(0, \sigma^2_g K)$
- $K$：**基因关系矩阵（GRM, genetic relationship matrix / kinship matrix）**，$K_{ij}$ 衡量个体 $i, j$ 的遗传相似度
- $\varepsilon \sim \mathcal{N}(0, \sigma^2_e I)$：独立噪声

**LMM 的效果**：通过 $g$ 捕捉群体结构和亲缘关系引起的相关性，等价于对联合分布做协方差校正。

> 精确 LMM 需要计算 $K^{-1}$（$O(n^3)$ 操作），在数万样本的 GWAS 中代价极高。
> 岭回归提供了一个高效近似。

---

### 2 · 全基因组岭回归 Whole-Genome Ridge Regression

**关键思路**：对 $G \in \mathbb{R}^{n \times m}$（标准化基因型）做岭回归，
用所有 SNP 的线性组合来预测 $y$——这个预测值 $\hat{y}_{\text{ridge}}$ 等价于 $g$ 的最大后验估计（MAP estimate）：

$$\hat{\beta} = (G_s^\top G_s + \alpha I)^{-1} G_s^\top (y - \bar{y})$$

$$\hat{y}_{\text{ridge}} = \bar{y} + G_s \hat{\beta}$$

其中 $\alpha$ 是**正则化参数（regularization parameter）**，对应 LMM 中方差比 $\sigma^2_e / \sigma^2_g$。

**计算技巧**（Woodbury）：当 $m > n$ 时（SNP 数远多于样本数，GWAS 常见），
可利用矩阵恒等式将 $m \times m$ 矩阵求逆转化为 $n \times n$ 矩阵求逆：

$$\hat{\beta} = G_s^\top (G_s G_s^\top + \alpha I)^{-1} (y - \bar{y})$$

这将计算复杂度从 $O(m^3)$ 降至 $O(n^3)$（$n \ll m$ 时显著加速）。

---

### 3 · 邻近污染与 LOCO Proximal Contamination & LOCO

**问题**：若用**所有** SNP（包括待检 SNP 所在染色体的 SNP）做岭回归预测，
则待检 SNP 的信号已被"解释掉"一部分，导致检验功效（power）下降。
这就是**邻近污染（proximal contamination）**。

**LOCO（留一染色体，Leave-One-Chromosome-Out）解决方案**：
检验**第 $c$ 条染色体**上的 SNP 时，只用**其他所有染色体**的 SNP 做岭回归，
得到**排除第 $c$ 染色体后的多基因预测值** $\hat{y}_{-c}$，
然后用残差 $\tilde{y}_c = y - \hat{y}_{-c}$ 作为表型进行关联测试：

$$\tilde{y}_c = y - \hat{y}_{\text{ridge, -c}}$$

如此，第 $c$ 染色体上的真实信号不会被多基因成分"吸收"。

---

### 4 · REGENIE 两步法 REGENIE Two-Step

**REGENIE**（Mbatchou et al. 2021, *Nature Genetics*）是当前 UK Biobank 规模 GWAS 的主流工具，核心是：

**Step 1（whole-genome regression）**：
1. 把基因型矩阵分成固定大小的块（block）
2. 在每块内做岭回归，把块级预测值汇总
3. 用所有块的汇总预测值做第二层岭回归（ridge on ridge）
4. 对每条染色体执行 LOCO，得到 per-chromosome 表型残差

**Step 2（SNP-level association）**：
用 Step 1 的 LOCO 残差作为"已校正表型"，对每个 SNP 逐一做 logistic / linear 回归（Firth 校正）。

**优势**：
- Step 1 的时间复杂度与样本量近似线性（分块）
- 支持大规模二元表型（binary trait）
- 自动处理亲缘关系和群体分层


## 示例 Worked Example

实现 `ridge_predict` 和 `loco_residualize`，并进行数值验证。

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os

rng = np.random.default_rng(42)
tmpdir = tempfile.mkdtemp()

# ── 合成多基因数据 ────────────────────────────────────────────
# n=200 样本, m=500 SNP, 其中 50 个 causal SNP 有真实效应
n, m = 200, 500
n_causal = 50

aaf = rng.uniform(0.1, 0.9, m)
G = rng.binomial(2, aaf, (n, m)).astype(float)

# 真实效应量（多基因架构：每个 causal SNP 效应 ~ N(0, h2/n_causal)）
h2 = 0.5           # 遗传力（heritability）
beta_true = np.zeros(m)
causal_idx = rng.choice(m, n_causal, replace=False)
beta_true[causal_idx] = rng.normal(0, np.sqrt(h2 / n_causal), n_causal)

# 表型 = 遗传成分 + 环境噪声
Gs = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)
genetic_comp = Gs @ beta_true
noise = rng.normal(0, np.sqrt(1 - h2), n)
y = genetic_comp + noise
y -= y.mean()  # 去均值

print(f"G.shape = {G.shape},  y.shape = {y.shape}")
print(f"遗传力 h2 ≈ {h2},  causal SNP = {n_causal}")
print(f"遗传成分标准差: {genetic_comp.std():.4f}")
print(f"噪声标准差:     {noise.std():.4f}")
print(f"表型方差:       {y.var():.4f}")


In [ ]:
# ── 实现 ridge_predict（镜像 b8-ridge-lmm）─────────────────
def ridge_predict(G, y, alpha=1.0):
    """全基因组岭回归预测：返回预测值 (n,)。
    G 先标准化；y 去均值后闭式解 beta=(Gs^T Gs + alpha*I)^{-1} Gs^T (y-ybar)。
    预测 = ybar + Gs @ beta。
    """
    G = np.asarray(G, dtype=float)
    y = np.asarray(y, dtype=float)
    Gs = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)
    ybar = y.mean()
    m_loc = Gs.shape[1]
    beta = np.linalg.solve(Gs.T @ Gs + alpha * np.eye(m_loc), Gs.T @ (y - ybar))
    return ybar + Gs @ beta


# 验证：与 sklearn Ridge 对比
alpha = 1.0
y_pred_ours = ridge_predict(G, y, alpha=alpha)

Gs_ref = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)
ridge_sk = Ridge(alpha=alpha, fit_intercept=True).fit(Gs_ref, y)
y_pred_sk = ridge_sk.predict(Gs_ref)

diff = np.abs(y_pred_ours - y_pred_sk).max()
r2_corr = np.corrcoef(y_pred_ours, y_pred_sk)[0, 1]

print(f"ridge_predict vs sklearn Ridge:")
print(f"  最大绝对差异 = {diff:.2e}  (应 < 1e-8)")
print(f"  预测值相关系数 = {r2_corr:.10f}  (应 = 1.0)")
print()

# 预测性能（与真实遗传成分的相关性）
r_genetic = np.corrcoef(y_pred_ours - y_pred_ours.mean(),
                         genetic_comp - genetic_comp.mean())[0, 1]
print(f"预测值与真实遗传成分的相关性 r = {r_genetic:.4f}")
print(f"（遗传力 h2={h2}，样本量 n={n}，SNP 数 m={m}，相关性反映预测精度）")


In [ ]:
# ── 可视化 ridge 预测效果 ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

ax = axes[0]
ax.scatter(genetic_comp, y_pred_ours, s=12, alpha=0.5, color="steelblue")
ax.set_xlabel("真实遗传成分 $g$")
ax.set_ylabel(r"岭回归预测值 $\hat{y}$")
ax.set_title(f"岭回归预测 vs 真实遗传成分\nr={r_genetic:.3f}")
lim = max(abs(genetic_comp).max(), abs(y_pred_ours).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], "r--", lw=0.8, label="y=x")
ax.legend(fontsize=8)

# 不同 alpha 的预测误差
alphas = np.logspace(-2, 3, 30)
mse_vals = []
for a in alphas:
    yp = ridge_predict(G, y, alpha=a)
    mse_vals.append(np.mean((yp - genetic_comp) ** 2))

ax2 = axes[1]
ax2.semilogx(alphas, mse_vals, "o-", color="steelblue", markersize=4)
ax2.set_xlabel(r"正则化参数 $\alpha$")
ax2.set_ylabel("预测 MSE（vs 真实遗传成分）")
ax2.set_title("Alpha 调参（不同正则化强度）")
best_alpha = alphas[np.argmin(mse_vals)]
ax2.axvline(best_alpha, color="red", linestyle="--", label=f"最优 alpha={best_alpha:.2f}")
ax2.legend(fontsize=8)

fig.tight_layout()
out = os.path.join(tmpdir, "ridge_pred.png")
fig.savefig(out, dpi=120)
plt.close(fig)
print(f"Ridge 预测图已保存：{out}")
print(f"最优 alpha = {best_alpha:.4f}")


In [ ]:
# ── 实现 loco_residualize（镜像 b8-loco）──────────────────
def loco_residualize(G, y, chrom, target_chrom, alpha=1.0):
    """留一染色体 LOCO：用除 target_chrom 外所有染色体的 SNP 做岭回归预测 y。
    返回残差 (n,)。检验 target_chrom 上的变异时用此残差作表型。"""
    G = np.asarray(G, dtype=float)
    y = np.asarray(y, dtype=float)
    chrom = np.asarray(chrom)
    other = G[:, chrom != target_chrom]
    Gs = (other - other.mean(axis=0)) / (other.std(axis=0) + 1e-8)
    m_other = Gs.shape[1]
    beta = np.linalg.solve(Gs.T @ Gs + alpha * np.eye(m_other), Gs.T @ y)
    return y - Gs @ beta


# 分配每个 SNP 到染色体（22 条常染色体）
n_chrom = 22
chrom_sizes = [max(1, int(m * (1 / n_chrom)))] * n_chrom
chrom_sizes[-1] += m - sum(chrom_sizes)
chrom_labels = np.concatenate([np.full(sz, c + 1) for c, sz in enumerate(chrom_sizes)])

print(f"SNP 染色体分配：{n_chrom} 条染色体，每条约 {chrom_sizes[0]} 个 SNP")
print(f"chrom_labels 范围: {chrom_labels.min()} ~ {chrom_labels.max()}")

# 计算 LOCO 残差（针对染色体 1）
target = 1
resid_loco = loco_residualize(G, y, chrom_labels, target_chrom=target, alpha=1.0)

print(f"\nLOCO 残差（排除染色体 {target}）:")
print(f"  shape = {resid_loco.shape}")
print(f"  均值 = {resid_loco.mean():.6f}  (应接近 0)")
print(f"  方差 = {resid_loco.var():.6f}")
print(f"  与原始 y 的相关性 = {np.corrcoef(y, resid_loco)[0,1]:.6f}")


In [ ]:
# ── 演示 LOCO 对邻近污染的抑制 ────────────────────────────
# 设置：causal SNP 在染色体 1 上，观察 LOCO vs 非 LOCO 的检验结果

# 找一个染色体 1 上的 causal SNP
chrom1_causal = [i for i in causal_idx if chrom_labels[i] == 1]
if len(chrom1_causal) == 0:
    chrom1_causal = np.where(chrom_labels == 1)[0][:1].tolist()

test_snp = chrom1_causal[0]
g_test = G[:, test_snp]

def single_snp_t(g, y_resid):
    """单 SNP t 统计量（去均值后简单回归）。"""
    g = g - g.mean()
    y_r = y_resid - y_resid.mean()
    beta = (g @ y_r) / (g @ g)
    n_l = len(y_r)
    resid = y_r - g * beta
    sigma2 = resid @ resid / (n_l - 2)
    se = np.sqrt(sigma2 / (g @ g))
    return beta / se

# 情景 1：用原始 y（无多基因校正）
t_raw = single_snp_t(g_test, y)

# 情景 2：用全部 SNP 岭回归残差（含邻近污染——包括了 target chrom）
y_ridge_all = ridge_predict(G, y, alpha=1.0)
resid_all = y - y_ridge_all
t_proximal = single_snp_t(g_test, resid_all)

# 情景 3：LOCO 残差（正确做法，无邻近污染）
resid_loco_c1 = loco_residualize(G, y, chrom_labels, target_chrom=1, alpha=1.0)
t_loco = single_snp_t(g_test, resid_loco_c1)

print("=== 邻近污染演示（染色体 1 上的 causal SNP） ===")
print(f"SNP index = {test_snp},  真实 beta = {beta_true[test_snp]:.4f}")
print()
print(f"情景 1 - 无多基因校正（原始 y）:        |t| = {abs(t_raw):.4f}")
print(f"情景 2 - 全染色体岭回归残差（邻近污染）: |t| = {abs(t_proximal):.4f}  (功效损失)")
print(f"情景 3 - LOCO 残差（正确 LOCO）:        |t| = {abs(t_loco):.4f}  (功效恢复)")
print()
print("LOCO 应比邻近污染情景有更高的 |t|，功效不受损")

# LOCO 应该：比 proximal 更强（功效恢复）
if abs(t_loco) > abs(t_proximal):
    print("验证通过：LOCO 比邻近污染情景功效更高")
else:
    print("（注：具体大小取决于随机数种子和数据，理论上 LOCO 应更好）")


In [ ]:
# ── 可视化：LOCO 残差分布与染色体分布 ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# (a) Ridge 预测 vs y
ax = axes[0]
ax.scatter(y, y_ridge_all, s=8, alpha=0.4, color="steelblue")
ax.set_xlabel("真实表型 $y$")
ax.set_ylabel(r"岭回归预测 $\hat{y}$")
ax.set_title("岭回归预测值")
lim_v = max(abs(y).max(), abs(y_ridge_all).max()) * 1.05
ax.plot([-lim_v, lim_v], [-lim_v, lim_v], "r--", lw=0.8)

# (b) LOCO 残差 vs 原始 y
ax2 = axes[1]
ax2.scatter(y, resid_loco_c1, s=8, alpha=0.4, color="seagreen")
ax2.set_xlabel("原始表型 $y$")
ax2.set_ylabel(r"LOCO 残差 $\tilde{y}_1$（排除染色体 1）")
ax2.set_title("LOCO 残差（染色体 1）")
ax2.axhline(0, color="gray", lw=0.8)

# (c) 每条染色体的 LOCO 残差方差
resid_vars = []
for c in range(1, n_chrom + 1):
    r = loco_residualize(G, y, chrom_labels, target_chrom=c, alpha=1.0)
    resid_vars.append(r.var())

ax3 = axes[2]
ax3.bar(range(1, n_chrom + 1), resid_vars, color="steelblue", edgecolor="white")
ax3.set_xlabel("染色体编号")
ax3.set_ylabel("LOCO 残差方差")
ax3.set_title("各染色体 LOCO 残差方差")
ax3.axhline(y.var(), color="red", linestyle="--", lw=0.8, label="原始 y 方差")
ax3.legend(fontsize=8)

fig.tight_layout()
out = os.path.join(tmpdir, "loco_demo.png")
fig.savefig(out, dpi=120)
plt.close(fig)
print(f"LOCO 演示图已保存：{out}")
print(f"原始 y 方差: {y.var():.4f}")
print(f"LOCO 残差方差范围: {min(resid_vars):.4f} ~ {max(resid_vars):.4f}")
print("（LOCO 残差方差约为 y 方差的 (1 - h2)，多基因遗传成分被校正）")


## 动手 Exercise

以下合成数据 `G_ex`、`y_ex`、`chrom_ex` 已准备好。请你：

1. 用 `ridge_predict(G_ex, y_ex, alpha=0.5)` 得到预测值，计算预测值与 `y_ex` 的 Pearson 相关系数（$R^2 = r^2$，量化岭回归拟合效果）。
2. 对染色体 3 做 LOCO 残差化：调用 `loco_residualize(G_ex, y_ex, chrom_ex, target_chrom=3, alpha=0.5)`，
   打印残差的方差，并与原始 `y_ex` 方差对比。
3. **验证 LOCO 特性**：对染色体 3 上 SNP 0 做 `single_snp_t` 检验，
   分别用（a）原始 `y_ex`，（b）`loco_residualize` 残差，打印 $|t|$ 值。

```python
rng_ex = np.random.default_rng(1729)
n_ex, m_ex = 150, 300
aaf_ex = rng_ex.uniform(0.1, 0.9, m_ex)
G_ex = rng_ex.binomial(2, aaf_ex, (n_ex, m_ex)).astype(float)
chrom_ex = np.concatenate([np.full(m_ex // 5, c + 1) for c in range(5)])
chrom_ex = np.append(chrom_ex, np.full(m_ex - len(chrom_ex), 5))
beta_ex = np.zeros(m_ex)
beta_ex[np.where(chrom_ex == 3)[0][:5]] = 0.3
Gs_ex = (G_ex - G_ex.mean(0)) / (G_ex.std(0) + 1e-8)
y_ex = Gs_ex @ beta_ex + rng_ex.normal(0, 1, n_ex)
y_ex -= y_ex.mean()

# 你的代码写在这里
```

> **提示**：LOCO 残差的方差应小于原始 `y_ex` 方差（多基因成分被移除），且染色体 3 的 causal SNP 应在 LOCO 残差上有更高的 $|t|$ 值（功效不受邻近污染影响）。


## 延伸阅读 Further Reading

1. **Yang et al. (2011)**. "GCTA: A tool for genome-wide complex trait analysis." *American Journal of Human Genetics.* LMM/GREML 方法的核心论文。
2. **Mbatchou et al. (2021)**. "Computationally efficient whole-genome regression for quantitative and binary traits." *Nature Genetics.* REGENIE 原始论文，详细介绍两步岭回归。
3. **Loh et al. (2015)**. "Efficient Bayesian mixed-model analysis increases association power in large cohorts." *Nature Genetics.* BOLT-LMM 的关键思路。
4. **Hoerl & Kennard (1970)**. "Ridge regression: Biased estimation for nonorthogonal problems." *Technometrics.* 岭回归的经典统计学论文。


## 关联练习 Related Assignments

### b8-ridge-lmm：全基因组岭回归

完成 `progress/<你>/work/b8-ridge-lmm/solution.py`，实现 `ridge_predict(G, y, alpha=1.0)` 函数。

评测命令：

```bash
python check.py 02-ridge-lmm
```

### b8-loco：留一染色体残差化

完成 `progress/<你>/work/b8-loco/solution.py`，实现 `loco_residualize(G, y, chrom, target_chrom, alpha=1.0)` 函数。

评测命令：

```bash
python check.py 03-loco
```

> 两个作业的函数签名与本课示例完全一致，可直接参照上方代码实现。
